# Pneumonia Detection from Chest X-Rays — CNN Pipeline
Binary classification: **NORMAL vs PNEUMONIA** using TensorFlow/Keras.

## 1. Imports & GPU Check

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras import Sequential, layers
from sklearn.metrics import confusion_matrix, classification_report

# Add project root to path so src.* imports work from the notebook
sys.path.insert(0, os.path.abspath('..'))

# GPU check — TF will use GPU automatically if available
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'None — running on CPU')

## 2. Configuration

In [ ]:
DATA_DIR   = '../data/raw/chest_xray'
TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
VAL_DIR    = os.path.join(DATA_DIR, 'val')
TEST_DIR   = os.path.join(DATA_DIR, 'test')
MODEL_PATH = '../models/pneumonia_cnn.keras'
PLOTS_DIR  = '../results/plots'

os.makedirs('../models', exist_ok=True)
os.makedirs(PLOTS_DIR,   exist_ok=True)

IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 1e-3
CLASSES     = ['NORMAL', 'PNEUMONIA']

print('Dataset folders:', os.listdir(DATA_DIR))

## 3. Data Preprocessing & Augmentation (ImageDataGenerator)

In [ ]:
# Training: augmentation + rescale
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation / Test: only rescale (no augmentation)
eval_datagen = ImageDataGenerator(rescale=1./255)

common = dict(target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
              color_mode='grayscale', class_mode='binary')

train_gen = train_datagen.flow_from_directory(TRAIN_DIR, shuffle=True,  **common)
val_gen   = eval_datagen.flow_from_directory(VAL_DIR,   shuffle=False, **common)
test_gen  = eval_datagen.flow_from_directory(TEST_DIR,  shuffle=False, **common)

print(f'Train samples : {train_gen.samples}')
print(f'Val samples   : {val_gen.samples}')
print(f'Test samples  : {test_gen.samples}')
print(f'Class indices : {train_gen.class_indices}')

## 4. Visualise Sample Images

In [ ]:
images, labels = next(train_gen)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, img, lbl in zip(axes.ravel(), images[:10], labels[:10]):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(CLASSES[int(lbl)], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Training Images (after augmentation)', y=1.01)
plt.tight_layout()
plt.show()

## 5. CNN Architecture
4 Conv blocks (32→64→128→256 filters) with BatchNorm + MaxPool,
followed by GlobalAveragePooling and a Dense head with Dropout.

In [ ]:
def build_model():
    model = Sequential([
        # Block 1
        layers.Conv2D(32, 3, padding='same', activation='relu',
                      input_shape=(*IMAGE_SIZE, 1)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 2
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 3
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 4
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Classifier head — GlobalAveragePooling avoids spatial overfitting
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid'),   # binary output
    ], name='pneumonia_cnn')
    return model

model = build_model()
model.summary()

## 6. Compile

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 7. Callbacks

In [ ]:
callbacks = [
    # Stop early if val_loss stops improving; restore the best weights
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    # Halve LR when val_loss plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    # Save the best checkpoint by val_accuracy
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

## 8. Train

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (train_key, val_key), title in zip(
    axes,
    [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
    ['Accuracy', 'Loss']
):
    ax.plot(history.history[train_key], label=f'Train {title}')
    ax.plot(history.history[val_key],   label=f'Val {title}')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 10. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f'\nTest Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.4f}')

## 11. Confusion Matrix, Precision, Recall & F1-Score

In [ ]:
test_gen.reset()
y_true = test_gen.classes
y_prob = model.predict(test_gen, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print('\n── Classification Report ──────────────────────────────────────')
print(classification_report(y_true, y_pred, target_names=CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 12. Predict on a Single Image

In [ ]:
def predict_single(img_path: str, threshold: float = 0.5):
    """Load one chest X-ray and return the predicted class + confidence."""
    img = load_img(img_path, color_mode='grayscale', target_size=IMAGE_SIZE)
    arr = img_to_array(img) / 255.0          # rescale to [0, 1]
    arr = np.expand_dims(arr, axis=0)        # add batch dimension → (1, 224, 224, 1)

    prob = model.predict(arr, verbose=0)[0][0]
    label = CLASSES[int(prob >= threshold)]

    plt.imshow(img, cmap='gray')
    plt.title(f'Prediction: {label}  (confidence: {prob:.2%})', fontsize=12)
    plt.axis('off')
    plt.show()

    return label, float(prob)


# ── Example usage ─────────────────────────────────────────────────────────────
# Replace the path below with any chest X-ray image on your machine.
SAMPLE_IMAGE = os.path.join(TEST_DIR, 'PNEUMONIA',
                             os.listdir(os.path.join(TEST_DIR, 'PNEUMONIA'))[0])

label, confidence = predict_single(SAMPLE_IMAGE)
print(f'Result → {label}  |  Confidence: {confidence:.2%}')

## 13. Save / Load Model

In [ ]:
# Best checkpoint was already saved by ModelCheckpoint callback.
# To reload it later:
# loaded_model = tf.keras.models.load_model(MODEL_PATH)
print(f'Best model saved at: {MODEL_PATH}')